# multi_4h_strat_026

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (202).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 8 |
| Detected functions | resample_1hr_custom, compute_atr, compute_jma_signals, prepare_backtest_data, backtest_single_ticker, run_backtest, rolling_r2, rogers_satchell_vol, calculate_metrics_fast |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2021-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from tqdm import tqdm


# ============================================================================
# TECHNICAL INDICATORS
# ============================================================================

def compute_atr(high, low, close, period=14):
    """Calculate Average True Range efficiently"""
    tr1 = high - low
    tr2 = np.abs(high - np.roll(close, 1))
    tr3 = np.abs(low - np.roll(close, 1))

    tr = np.maximum(tr1, np.maximum(tr2, tr3))
    tr[0] = np.nan  # First value invalid due to shift

    return pd.Series(tr).rolling(period, min_periods=1).mean().values


def compute_jma_signals(df, smooth_len=14, jma_norm_period=60, lower_thresh=-0.75,
                        rs_norm_period=40, rs_period=35, rs_threshold=0.35):
    """
    Compute JMA-based signals with Rogers-Satchell volatility filter

    Returns DataFrame with columns: JMA, Normalized_JMA, RS_Volatility, Signal
    """
    result = df.copy()
    price = result['close'].shift(1)

    # --- JMA (Exponential Moving Average) ---
    alpha = 2.0 / (smooth_len + 1)
    jma = price.ewm(alpha=alpha, adjust=False).mean()

    # --- Normalize JMA ---
    jma_rolling = jma.rolling(jma_norm_period)
    norm_jma = (jma - jma_rolling.mean()) / jma_rolling.std()

    # --- Rogers-Satchell Volatility ---
    o = result['open'].shift(1).replace(0, np.nan)
    h = result['high'].shift(1)
    l = result['low'].shift(1)
    c = result['close'].shift(1).replace(0, np.nan)

    rs_vol = (np.log(h/c) * np.log(h/o) + np.log(l/c) * np.log(l/o)).rolling(rs_period).mean() * 1e6
    rs_norm = rs_vol.rolling(rs_norm_period).apply(lambda x: x.rank(pct=True).iloc[-1])

    # --- Generate Signals ---
    signal = np.where((norm_jma < lower_thresh) & (rs_norm > rs_threshold), 2, 0)

    result['Signal'] = signal
    return result


def prepare_backtest_data(data, atr_mult=0.1):
    """
    Prepare all indicators and signals for backtesting

    Process per ticker:
    - JMA signals with RS volatility filter
    - ATR indicators (14 & 5 period)
    - ATR_max for stop loss calculation
    """
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    processed = []

    for ticker, group in tqdm(data.groupby('Ticker'), desc="Preparing data"):
        # Reset index for proper array operations
        group = group.reset_index(drop=True)

        # Compute signals
        group = compute_jma_signals(group)

        # Compute ATRs (vectorized)
        group['ATR_14'] = compute_atr(group['high'].values, group['low'].values,
                                       group['close'].values, period=14)
        group['ATR_5'] = compute_atr(group['high'].values, group['low'].values,
                                      group['close'].values, period=5)

        # Compute ATR_max (max of rolling max of both ATRs)
        atr14_max = pd.Series(group['ATR_14']).rolling(20).max() * atr_mult
        atr5_max = pd.Series(group['ATR_5']).rolling(20).max() * atr_mult
        group['ATR_max'] = np.maximum(atr14_max, atr5_max)

        # Shift signal for next-bar execution (avoid lookahead bias)
        group['Signal'] = group['Signal'].shift(1)

        processed.append(group)

    result = pd.concat(processed, ignore_index=True)
    result = result.dropna(subset=['Signal'])
    result['Signal'] = result['Signal'].astype(np.int8)

    return result


# ============================================================================
# BACKTEST ENGINE (Numba-optimized)
# ============================================================================

@njit
def backtest_single_ticker(dates, opens, highs, lows, closes, signals, atrs,
                          capital, fee_rate, slip_rate, entry_sl_pct):
    """
    Fast backtest for single ticker using Numba JIT compilation

    Strategy: Short-only with trailing stop loss
    Entry: Signal == 2
    Exit: Trailing stop (previous close + ATR_max)
    """
    n = len(dates)

    # Trade storage (pre-allocate for speed)
    max_trades = n  # Worst case: one trade per bar
    entry_times = np.empty(max_trades, dtype=np.float64)
    exit_times = np.empty(max_trades, dtype=np.float64)
    entry_prices = np.empty(max_trades, dtype=np.float64)
    exit_prices = np.empty(max_trades, dtype=np.float64)
    pnls = np.empty(max_trades, dtype=np.float64)
    cum_pnls = np.empty(max_trades, dtype=np.float64)
    exit_reasons = np.empty(max_trades, dtype=np.int8)  # 0=Entry SL, 1=Trailing SL

    trade_count = 0
    cum_pnl = 0.0

    # Position state
    in_position = False
    entry_price = 0.0
    sl_price = 0.0
    qty = 0.0
    entry_idx = 0
    last_exit_idx = -1
    prev_close = 0.0
    prev_atr = 0.0

    for i in range(n):
        # --- ENTRY LOGIC ---
        if not in_position and i != last_exit_idx and signals[i] == 2:
            # Calculate entry with slippage
            entry_price = opens[i] * (1.0 - slip_rate)
            qty = capital / entry_price
            sl_price = entry_price * (1.0 + entry_sl_pct)

            entry_idx = i
            in_position = True

            # Check immediate stop loss on entry bar
            if highs[i] >= sl_price:
                exit_price = sl_price * (1.0 + slip_rate)
                fees = fee_rate * (entry_price + exit_price) * qty
                pnl = (entry_price - exit_price) * qty - fees
                cum_pnl += pnl

                # Record trade
                entry_times[trade_count] = dates[i]
                exit_times[trade_count] = dates[i]
                entry_prices[trade_count] = entry_price
                exit_prices[trade_count] = exit_price
                pnls[trade_count] = pnl
                cum_pnls[trade_count] = cum_pnl
                exit_reasons[trade_count] = 0  # Entry SL
                trade_count += 1

                in_position = False
                last_exit_idx = i

            prev_close = closes[i]
            prev_atr = atrs[i]
            continue

        # --- POSITION MANAGEMENT ---
        if in_position and i != entry_idx:
            # Update trailing stop loss
            sl_price = min(sl_price, prev_close + prev_atr)

            # Check if stop loss hit
            if highs[i] >= sl_price:
                exit_price = sl_price * (1.0 + slip_rate)
                fees = fee_rate * (entry_price + exit_price) * qty
                pnl = (entry_price - exit_price) * qty - fees
                cum_pnl += pnl

                # Record trade
                entry_times[trade_count] = dates[entry_idx]
                exit_times[trade_count] = dates[i]
                entry_prices[trade_count] = entry_price
                exit_prices[trade_count] = exit_price
                pnls[trade_count] = pnl
                cum_pnls[trade_count] = cum_pnl
                exit_reasons[trade_count] = 1  # Trailing SL
                trade_count += 1

                in_position = False
                last_exit_idx = i

        prev_close = closes[i]
        prev_atr = atrs[i]

    # Return only filled trades
    return (entry_times[:trade_count], exit_times[:trade_count],
            entry_prices[:trade_count], exit_prices[:trade_count],
            pnls[:trade_count], cum_pnls[:trade_count],
            exit_reasons[:trade_count])


def run_backtest(data, entry_sl_pct=0.02, capital=100000,
                fee_rate=0.0005, slip_rate=0.00025):
    """
    Run backtest across all tickers and aggregate results

    Parameters:
    -----------
    entry_sl_pct : float
        Initial stop loss percentage (e.g., 0.02 = 2%)
    capital : float
        Position size in dollars
    fee_rate : float
        Trading fee as decimal (e.g., 0.0005 = 0.05%)
    slip_rate : float
        Slippage as decimal (e.g., 0.00025 = 0.025%)
    """
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    all_trades = []

    for ticker in tqdm(data['Ticker'].unique(), desc="Backtesting"):
        ticker_data = data[data['Ticker'] == ticker].reset_index(drop=True)

        # Convert dates to numeric for Numba
        dates = ticker_data['Date'].astype(np.int64).values / 1e9  # Convert to Unix timestamp

        # Run optimized backtest
        results = backtest_single_ticker(
            dates,
            ticker_data['open'].values,
            ticker_data['high'].values,
            ticker_data['low'].values,
            ticker_data['close'].values,
            ticker_data['Signal'].values,
            ticker_data['ATR_max'].values,
            capital, fee_rate, slip_rate, entry_sl_pct
        )

        if len(results[0]) > 0:  # If trades exist
            # Convert back to DataFrame
            trades_df = pd.DataFrame({
                'Ticker': ticker,
                'Entry Time': pd.to_datetime(results[0], unit='s'),
                'Exit Time': pd.to_datetime(results[1], unit='s'),
                'Entry Price': results[2],
                'Exit Price': results[3],
                'PnL': results[4],
                'Exit Reason': ['Entry SL' if r == 0 else 'Trailing SL' for r in results[6]],
                'Cumulative PnL': results[5],
                'Direction': 'short'
            })
            all_trades.append(trades_df)

    if all_trades:
        return pd.concat(all_trades, ignore_index=True)
    else:
        return pd.DataFrame()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Step 1: Prepare data with indicators
    print("Preparing data...")
    data = prepare_backtest_data(data, atr_mult=0.1)

    # Step 2: Run backtest
    print("Running backtest...")
    tradebook = run_backtest(
        data,
        entry_sl_pct=0.02,      # 2% initial stop loss
        capital=100000,          # $100k per position
        fee_rate=0.0005,        # 0.05% trading fee
        slip_rate=0.00025       # 0.025% slippage
    )

    # Step 3: Save results
    print(f"Total trades: {len(tradebook)}")
    tradebook.to_csv("/content/drive/MyDrive/tradebook_BTCUSDT_Q7.csv", index=False)

tradebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from itertools import product
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def rolling_r2(series, window=30):
    """Vectorized rolling R² calculation"""
    r2 = np.full(len(series), np.nan)
    x = np.arange(window).reshape(-1, 1)

    for i in range(window, len(series)):
        y = series.iloc[i - window:i].values.reshape(-1, 1)
        model = LinearRegression().fit(x, y)
        r2[i] = model.score(x, y)

    return pd.Series(r2, index=series.index)

def rogers_satchell_vol(df, rs_period):
    """Vectorized Rogers-Satchell volatility"""
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    rs = np.log(h/c) * np.log(h/o) + np.log(l/c) * np.log(l/o)
    return rs.rolling(rs_period).mean() * 1e6

def calculate_metrics_fast(true_trend, rs_active):
    """Optimized metric calculation using numpy"""
    true_trend_arr = true_trend.values
    rs_active_arr = rs_active.values

    # Trend recall
    true_trend_sum = np.sum(true_trend_arr == 1)
    trend_recall = np.sum((rs_active_arr == 1) & (true_trend_arr == 1)) / max(1, true_trend_sum)

    # False trend rate
    rs_active_sum = np.sum(rs_active_arr == 1)
    false_trend_rate = np.sum((rs_active_arr == 1) & (true_trend_arr == 0)) / max(1, rs_active_sum)

    # Regime stability (vectorized)
    changes = np.sum(rs_active_arr[1:] != rs_active_arr[:-1])
    avg_duration = len(rs_active_arr) / max(1, changes)

    coverage = np.mean(rs_active_arr)

    return trend_recall, false_trend_rate, avg_duration, coverage

# Prepare data once
print("Preparing data...")
df = data.copy()
df['r2'] = rolling_r2(df['close'], 30)
df['true_trend'] = (df['r2'] > 0.6).astype(int)

# OPTIMIZED PARAMETER GRID (reduced for speed)
rs_periods = [15, 20, 25, 30, 35, 40, 45]
norm_periods = [30, 40, 50, 60, 70]  # Reduced from every 10
rs_thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
rs_persist_values = [2]

# HARD FILTERS
COVERAGE_MIN, COVERAGE_MAX = 0.20, 0.55
MIN_AVG_DURATION = 3.0
WEIGHT_RECALL, WEIGHT_FALSE_RATE = 0.6, 0.4

total_combinations = len(rs_periods) * len(norm_periods) * len(rs_thresholds) * len(rs_persist_values)
print(f"\nTesting {total_combinations:,} parameter combinations")
print(f"RS Periods: {len(rs_periods)}, Norm Periods: {len(norm_periods)}")
print(f"RS Thresholds: {len(rs_thresholds)}, RS Persist: {len(rs_persist_values)}\n")

# Pre-calculate all RS volatilities (major speedup)
print("Pre-calculating RS volatilities...")
rs_vols = {rs_p: rogers_satchell_vol(df, rs_p) for rs_p in tqdm(rs_periods, desc="RS Vol", ncols=80)}

results = []
param_combinations = list(product(rs_periods, norm_periods, rs_thresholds, rs_persist_values))

# Main optimization loop
for rs_p, norm_p, rs_th, rs_persist in tqdm(param_combinations, desc="Optimizing", ncols=100):
    # Use pre-calculated RS vol
    rs_vol = rs_vols[rs_p]

    # Normalize
    rs_norm = rs_vol.rolling(norm_p).apply(lambda x: x.rank(pct=True).iloc[-1], raw=False)

    # Apply persistence filter
    rs_active_raw = (rs_norm > rs_th).astype(int)
    rs_active = (rs_active_raw.rolling(rs_persist).sum() == rs_persist).astype(int)

    # Filter valid data
    valid = rs_norm.notna()
    if valid.sum() < 100:  # Skip if too little data
        continue

    temp_true_trend = df['true_trend'][valid]
    temp_rs_active = rs_active[valid]

    # Fast metrics
    trend_recall, false_trend_rate, avg_duration, coverage = calculate_metrics_fast(
        temp_true_trend, temp_rs_active
    )

    # Quick filters (fail fast)
    if not (COVERAGE_MIN <= coverage <= COVERAGE_MAX) or avg_duration < MIN_AVG_DURATION:
        continue

    # Score
    rs_score = WEIGHT_RECALL * trend_recall - WEIGHT_FALSE_RATE * false_trend_rate

    results.append({
        'rs_period': rs_p,
        'norm_period': norm_p,
        'rs_threshold': rs_th,
        'rs_persist': rs_persist,
        'score': rs_score,
        'trend_recall': trend_recall,
        'false_trend_rate': false_trend_rate,
        'avg_duration': avg_duration,
        'coverage': coverage
    })

# Results analysis
res_df = pd.DataFrame(results)
print(f"\n✓ Found {len(res_df):,} valid combinations ({len(res_df)/total_combinations*100:.1f}%)")

if len(res_df) == 0:
    print("No valid combinations. Relax filters.")
else:
    res_df.sort_values('score', ascending=False, inplace=True)

    print("\n" + "="*80)
    print("TOP 10 CONFIGURATIONS")
    print("="*80)
    print(res_df.head(10).to_string(index=False, float_format='%.4f'))

    # Best config
    best = res_df.iloc[0]
    print("\n" + "="*80)
    print("BEST CONFIGURATION")
    print("="*80)
    print(f"RS Period: {best['rs_period']}, Norm Period: {best['norm_period']}")
    print(f"RS Threshold: {best['rs_threshold']}, RS Persist: {best['rs_persist']}")
    print(f"Score: {best['score']:.4f} | Recall: {best['trend_recall']:.4f} | False Rate: {best['false_trend_rate']:.4f}")
    print(f"Avg Duration: {best['avg_duration']:.2f} | Coverage: {best['coverage']:.4f}")

    # Quick visualizations
    best_norm = int(res_df['norm_period'].mode()[0])
    best_persist = int(res_df['rs_persist'].mode()[0])
    df_plot = res_df[(res_df['norm_period'] == best_norm) & (res_df['rs_persist'] == best_persist)]

    if len(df_plot) > 5:
        pivot = df_plot.pivot_table(index='rs_period', columns='rs_threshold', values='score')

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Heatmap
        im = axes[0].imshow(pivot.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
        axes[0].set_xticks(range(len(pivot.columns)))
        axes[0].set_xticklabels([f'{x:.2f}' for x in pivot.columns])
        axes[0].set_yticks(range(len(pivot.index)))
        axes[0].set_yticklabels(pivot.index)
        plt.colorbar(im, ax=axes[0], label='Score')
        axes[0].set_xlabel('RS Threshold')
        axes[0].set_ylabel('RS Period')
        axes[0].set_title(f'Score Heatmap (norm={best_norm}, persist={best_persist})')

        # Score distribution
        axes[1].hist(res_df['score'], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
        axes[1].axvline(best['score'], color='green', linestyle='--', linewidth=2,
                       label=f'Best: {best["score"]:.4f}')
        axes[1].set_xlabel('Score')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Score Distribution')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Parameter sensitivity (2x2 grid)
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        params = [
            ('rs_period', 'RS Period', axes[0, 0]),
            ('norm_period', 'Norm Period', axes[0, 1]),
            ('rs_threshold', 'RS Threshold', axes[1, 0]),
            ('rs_persist', 'RS Persist', axes[1, 1])
        ]

        for param, label, ax in params:
            grouped = res_df.groupby(param)['score'].agg(['mean', 'max'])
            ax.plot(grouped.index, grouped['mean'], 'o-', label='Mean', markersize=6)
            ax.plot(grouped.index, grouped['max'], 's--', label='Max', markersize=5)
            ax.set_xlabel(label, fontsize=11)
            ax.set_ylabel('Score', fontsize=11)
            ax.set_title(f'{label} Sensitivity', fontsize=12)
            ax.legend()
            ax.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()